# Notebook 3 — Feature Engineering & ML Pipeline Construction
## HMDA 2023 Loan Denial Prediction: From Raw Data → Model-Ready Vectors

**Objective:** Transform 99 raw HMDA columns into a clean, encoded, scaled feature matrix that PySpark ML classifiers can consume directly.

---

### Why feature engineering is the highest-leverage ML activity

> *"Coming up with features is difficult, time-consuming, [and] requires expert knowledge.
> Applied machine learning is basically feature engineering."*
> — Andrew Ng

A well-engineered feature set with Logistic Regression routinely outperforms a complex neural network on raw features. Google's "Ad Click Prediction: a View from the Trenches" paper confirms that feature engineering dominates model choice for structured data.

In **HMDA specifically**, raw columns like `debt_to_income_ratio` are STRING bins (`"20%-<30%"`), `income` contains `"Exempt"` strings masquerading as missing values, and `hoepa_status` has near-zero variance — but its rare `1` value is one of the strongest denial signals. This notebook handles all of that.

---

### Blueprint — What we build & why (each step is driven by EDA findings from Notebook 2)

| Step | What | EDA Source | Why |
|------|------|-----------|-----|
| 1 | Column Name Normalization | — | Fix hyphen→underscore mismatches from Parquet |
| 2 | Column Selection (drop 55+ cols) | Cell 10, 21, 22 | Remove leakage, identifiers, NZV, redundant demographics |
| 3 | Target Creation (`label` 0/1) | Cell 11 | Keep only originated (1) & denied (3) |
| 4 | Missing Value Normalization | Cell 19-20 | "Exempt"/"NA"/1111 → null for Imputer compatibility |
| 5 | Missingness Indicators | Cell 20 | Informative missingness = free signal |
| 6 | Numeric Transforms | Cell 4-7 | Log-transform skew, clip outliers, create ratios |
| 7 | Categorical Encoding Plan | Cell 8-9 | Cardinality-driven: OneHot ≤10, StringIndex 11-50 |
| 8 | PySpark ML Pipeline Assembly | — | Imputer → StringIndexer → OHE → VectorAssembler → Scaler |
| 9 | Stratified Train/Test Split | Cell 11 | Preserve ~80/20 class imbalance |
| 10 | Pipeline Fit & Transform | — | fit(train), transform(both) — no data leakage |
| 11 | Verification & Save | — | Row counts, vector dims, class balance checks |

### Key PySpark ML Architecture

```
Raw DataFrame
    ↓  [Column name normalization, drops, missing normalization, feature engineering]
Clean DataFrame (all columns still separate)
    ↓  [Pipeline.fit(train)]
PipelineModel
    ↓  [PipelineModel.transform(train | test)]
Transformed DataFrame with "features" vector column + "label"
    ↓  [Save to Parquet]
Ready for Notebook 4: Model Training
```

## Section 1 — Setup & Data Loading

**WHY reload from Parquet (not from Notebook 2's cached DataFrame)?**
Each notebook must be independently runnable. In production, you can't assume a previous notebook's SparkSession is alive. Parquet loads in seconds regardless.

**WHY load the schema JSON and EDA pickle?**
- `hmda_schema.json` provides the ground-truth column classifications (numeric, categorical, leakage) that we defined in Notebook 1
- `eda_results.pkl` contains empirically-verified findings (NZV columns, informative missing patterns, multicollinear pairs) computed in Notebook 2

This ensures our feature engineering decisions are **data-driven**, not hardcoded guesses.

In [1]:
# ============================================================
# Cell 1: Imports & SparkSession
# ============================================================

# WHY each import group?
# ─── Core Spark ───
# SparkSession: Entry point for all Spark functionality
# functions (F): Column-level operations — Spark's equivalent of pandas methods
# types: Schema definitions for casting columns
#
# ─── PySpark ML ───
# StringIndexer: Maps categorical strings → numeric indices (e.g., "FHA"→1)
# OneHotEncoder: Converts indices → sparse binary vectors (e.g., 1→[0,1,0,0])
# VectorAssembler: Merges all feature columns → single "features" vector
# StandardScaler: Normalizes to zero mean / unit variance
# Imputer: Replaces null with median/mean (fit on train only!)
# Pipeline: Chains all above into a single reproducible workflow
#
# ─── Utilities ───
# pandas/numpy: For small verification tables, NOT for 10M-row processing

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType, IntegerType
from pyspark.sql.window import Window

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler,
    StandardScaler, Imputer
)
from pyspark.ml.stat import Summarizer

import pandas as pd
import numpy as np
import json
import pickle
import time
import warnings
warnings.filterwarnings('ignore')

# WHY these specific Spark configs?
# - 8g driver: Feature engineering creates many intermediate columns → more RAM needed
# - AQE: Adaptively optimizes shuffle partitions during Pipeline fitting
# - Arrow: 10-100x speedup for toPandas() in verification steps
# - maxResultSize 2g: VectorAssembler on 8M+ rows can produce large result objects

spark = (SparkSession.builder
    .appName("HMDA_2023_FeatureEngineering")
    .master("local[*]")
    .config("spark.driver.memory", "8g")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} | Cores: {spark.sparkContext.defaultParallelism}")

26/03/01 17:39:45 WARN Utils: Your hostname, Adityas-Laptop.local resolves to a loopback address: 127.0.0.1; using 10.247.13.101 instead (on interface en0)
26/03/01 17:39:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/01 17:39:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 3.5.8 | Cores: 8


In [2]:
# ============================================================
# Cell 2: Load Data, Schema & EDA Results
# ============================================================

# ── Paths (adjust for your local environment) ──
PARQUET_PATH   = "/Users/adi/Desktop/assignmt/project/data/processed/hmda_2023.parquet"
SCHEMA_PATH    = "/Users/adi/Desktop/assignmt/project/data/schemas/hmda_schema.json"
EDA_RESULTS    = "/Users/adi/Desktop/assignmt/project/data/processed/eda_results.pkl"
OUTPUT_DIR     = "/Users/adi/Desktop/assignmt/project/data/processed"

# ── Load Parquet ──
df = spark.read.parquet(PARQUET_PATH)
total_rows = df.count()
print(f"Loaded: {total_rows:,} rows × {len(df.columns)} columns")

# ── Load Schema ──
with open(SCHEMA_PATH, "r") as f:
    schema = json.load(f)

# Reference lists from schema — these are our ground-truth column classifications
NUMERIC_CONTINUOUS  = schema["expected_dtypes"]["numeric_continuous"]     # 22 cols
CATEGORICAL_NOMINAL = schema["expected_dtypes"]["categorical_nominal"]   # 67 cols
CATEGORICAL_ORDINAL = schema["expected_dtypes"]["categorical_ordinal"]   # 6 cols
LEAKAGE_COLS        = schema["leakage_columns"]["columns"]               # 12 cols

print(f"\nSchema loaded:")
print(f"  Numeric continuous:  {len(NUMERIC_CONTINUOUS)}")
print(f"  Categorical nominal: {len(CATEGORICAL_NOMINAL)}")
print(f"  Categorical ordinal: {len(CATEGORICAL_ORDINAL)}")
print(f"  Leakage columns:     {len(LEAKAGE_COLS)}")

# ── Load EDA Results (optional — graceful fallback) ──
try:
    with open(EDA_RESULTS, "rb") as f:
        eda = pickle.load(f)
    print(f"\nEDA results loaded: {len(eda)} keys")
    print(f"  NZV columns:         {len(eda.get('nzv_cols', []))}")
    print(f"  Multicollinear pairs:{len(eda.get('multicoll_pairs', []))}")
    print(f"  Informative missing: {len(eda.get('informative_missing', []))}")
except FileNotFoundError:
    print("\n⚠ EDA results not found — using schema-based defaults")
    eda = {}

Loaded: 11,483,889 rows × 99 columns

Schema loaded:
  Numeric continuous:  22
  Categorical nominal: 69
  Categorical ordinal: 6
  Leakage columns:     12

EDA results loaded: 12 keys
  NZV columns:         28
  Multicollinear pairs:1
  Informative missing: 60


## Section 1b — Column Name Normalization

> **🐛 BUG FIX (v2):** The original notebook defined drop lists with hyphenated names
> (e.g., `denial_reason-1`) while Spark/Parquet had already sanitized column names to
> underscores (`denial_reason_1`). This caused ~32 columns to silently escape the drop step,
> including **4 leakage columns** (`denial_reason_1` through `_4`).
>
> **Impact:** The final feature vector was 287 dimensions instead of ~170.
> Worse, the 4 surviving denial-reason columns constituted **data leakage** —
> they are only populated for denied applications, so the model was effectively
> told the answer during training.

**Fix:** We normalize ALL column names to a canonical form (lowercase, hyphens → underscores)
immediately after loading, BEFORE any drop logic runs. This guarantees name matching works.

In [3]:
# ============================================================
# Cell 2b: Normalize Column Names (hyphen → underscore)
# ============================================================
#
# 🐛 BUG FIX: HMDA raw CSV uses hyphens in some column names.
# Spark/Parquet may convert these to
# underscores on read. Our drop lists MUST match the actual DataFrame
# column names exactly — otherwise drops silently fail.
#
# We canonicalize everything to underscores here so the rest of the
# notebook never needs to worry about it.

original_cols = df.columns[:]
rename_count = 0

for col_name in original_cols:
    clean = col_name.replace("-", "_")
    if clean != col_name:
        df = df.withColumnRenamed(col_name, clean)
        rename_count += 1

print(f"✓ Normalized {rename_count} column names (hyphen → underscore)")
if rename_count > 0:
    print(f"  Examples of renamed columns:")
    for old in original_cols:
        new = old.replace("-", "_")
        if new != old:
            print(f"    {old:40s} → {new}")
            if rename_count > 8:
                break  # Don't flood output

print(f"\n  Total columns: {len(df.columns)}")

✓ Normalized 0 column names (hyphen → underscore)

  Total columns: 99


## Section 2 — Column Selection: Dropping Useless & Dangerous Columns

**WHY is this the very first step?**
Every column we keep costs compute: memory for caching, CPU for encoding, model complexity for training. Dropping columns BEFORE any transformation is the cheapest possible optimization — it shrinks the DataFrame immediately.

**Five categories to drop (justified by EDA Notebook 2):**

### A. Leakage Columns (12 cols) — EDA Cell 21: Empirically confirmed
These are populated **AFTER** the lending decision. Using them as features = using the answer to predict the answer.
- `denial_reason_1` through `_4`: Literally "why was this denied?" — only filled for denials
- `purchaser_type`: Who bought the loan — known only after origination
- `rate_spread`, `total_loan_costs`, etc.: Final loan terms, set after approval

### B. Identifier Columns (5 cols) — too many unique values to generalize
- `lei`: 5000+ unique lender IDs → model memorizes specific lenders, doesn't generalize
- `census_tract`: 70,000+ values → can't one-hot, codes don't transfer to new data
- `county_code`, `activity_year`, `derived_msa_md`: Same problem

### C. Near-Zero Variance (7+ cols) — EDA Cell 10: >95% single value
When `negative_amortization` is "2" in 99% of rows, its one-hot column would be 99% zeros — pure noise.

### D. Redundant Demographic Detail (28 cols)
`applicant_ethnicity_1` through `_5` decompose into 12 columns that are mostly null. The CFPB already consolidated them into `derived_ethnicity` (one clean column). Same logic for race and sex granular fields.

### E. Underwriting System Detail (4 cols)
`aus_2` through `aus_5` are >95% null. Only `aus_1` has meaningful fill rate.

**INTERVIEW TIP:** *"I dropped 55+ columns before any transformation: 12 leakage (empirically verified), 5 identifiers, 7 NZV, 28 redundant demographics, and 4 AUS detail fields. This cut the feature space by over 55% before spending any compute on encoding."*

In [4]:
# ============================================================
# Cell 3: Define All Columns to Drop
# ============================================================

# WHY explicit lists instead of dynamic rules?
# In production ML, your feature set MUST be deterministic.
# If a dynamic NZV threshold changes, your model silently uses
# different features — a debugging nightmare. Explicit lists
# are auditable, version-controlled, and reproducible.
#
# 🐛 FIX (v2): All column names now use UNDERSCORES to match the
# normalized DataFrame from Cell 2b. The original notebook used
# hyphens which silently failed to match the Parquet column names,
# leaving ~32 columns — including 4 LEAKAGE columns — in the data.

# A. LEAKAGE — empirically confirmed in EDA Cell 21
LEAKAGE_DROP = [
    "denial_reason_1", "denial_reason_2", "denial_reason_3", "denial_reason_4",
    "purchaser_type", "rate_spread",
    "total_loan_costs", "total_points_and_fees",
    "origination_charges", "discount_points", "lender_credits",
    "prepayment_penalty_term",
]

# B. IDENTIFIERS — too high cardinality, no predictive generalization
IDENTIFIER_DROP = [
    "lei",            # 5000+ unique lender IDs
    "census_tract",   # 70,000+ unique tract codes
    "county_code",    # 3000+ counties
    "activity_year",  # Single value (2023) — literally zero variance
    "derived_msa_md", # Metro area code — use census tract context cols instead
]

# C. NEAR-ZERO VARIANCE — >95% single value (EDA Cell 10)
NZV_DROP = [
    "reverse_mortgage",               # >99% "2" (No)
    "open_end_line_of_credit",        # >95% "2" (No)
    "business_or_commercial_purpose", # >95% "2" (No)
    "negative_amortization",          # >99% "2" (No)
    "interest_only_payment",          # >95% "2" (No)
    "balloon_payment",                # >99% "2" (No)
    "other_nonamortizing_features",   # >95% "2" (No)
    "hoepa_status",                   # >99% "2" — BUT we extract high_cost_flag FIRST
]

# D. REDUNDANT DEMOGRAPHIC DETAIL — use derived_* versions instead
DEMOGRAPHIC_DETAIL_DROP = [
    "applicant_ethnicity_1", "applicant_ethnicity_2", "applicant_ethnicity_3",
    "applicant_ethnicity_4", "applicant_ethnicity_5",
    "co_applicant_ethnicity_1", "co_applicant_ethnicity_2", "co_applicant_ethnicity_3",
    "co_applicant_ethnicity_4", "co_applicant_ethnicity_5",
    "applicant_ethnicity_observed", "co_applicant_ethnicity_observed",
    "applicant_race_1", "applicant_race_2", "applicant_race_3",
    "applicant_race_4", "applicant_race_5",
    "co_applicant_race_1", "co_applicant_race_2", "co_applicant_race_3",
    "co_applicant_race_4", "co_applicant_race_5",
    "applicant_race_observed", "co_applicant_race_observed",
    "applicant_sex", "co_applicant_sex",
    "applicant_sex_observed", "co_applicant_sex_observed",
]

# E. AUS DETAIL — aus_2 through aus_5 are >95% null
AUS_DETAIL_DROP = ["aus_2", "aus_3", "aus_4", "aus_5"]

# F. TARGET SOURCE — we derive `label` from it, then drop
TARGET_SOURCE_DROP = ["action_taken"]

# ── Combine all drops ──
ALL_DROP = list(set(
    LEAKAGE_DROP + IDENTIFIER_DROP + NZV_DROP +
    DEMOGRAPHIC_DETAIL_DROP + AUS_DETAIL_DROP + TARGET_SOURCE_DROP
))

# Only drop columns that actually exist (defensive coding)
existing_drops = [c for c in ALL_DROP if c in df.columns]
missing_drops  = [c for c in ALL_DROP if c not in df.columns]

print(f"Total columns in DataFrame:  {len(df.columns)}")
print(f"Columns to drop:             {len(existing_drops)}")
print(f"  Leakage:              {len([c for c in LEAKAGE_DROP if c in df.columns])}")
print(f"  Identifiers:          {len([c for c in IDENTIFIER_DROP if c in df.columns])}")
print(f"  Near-zero variance:   {len([c for c in NZV_DROP if c in df.columns])}")
print(f"  Demographic detail:   {len([c for c in DEMOGRAPHIC_DETAIL_DROP if c in df.columns])}")
print(f"  AUS detail:           {len([c for c in AUS_DETAIL_DROP if c in df.columns])}")
print(f"  Target source:        {len([c for c in TARGET_SOURCE_DROP if c in df.columns])}")
print(f"Columns remaining:           {len(df.columns) - len(existing_drops)}")

# ── 🐛 FIX (v2): ASSERT that critical drops are present ──
# The original notebook said "Not found (OK, already renamed)" — that was WRONG.
# If a column we intend to drop doesn't exist, that's a schema change or naming
# bug that needs human investigation, NOT a silent pass.

if missing_drops:
    print(f"\n⚠ WARNING: {len(missing_drops)} intended drop columns not found in DataFrame:")
    for c in sorted(missing_drops):
        print(f"    MISSING: {c}")
    print("\n  → Check if these were already dropped in a prior step or renamed.")
    print("  → If this is unexpected, investigate before proceeding!")

# Hard assert on leakage columns — these MUST be dropped
leakage_missing = [c for c in LEAKAGE_DROP if c not in df.columns]
assert len(leakage_missing) == 0, (
    f"CRITICAL: {len(leakage_missing)} LEAKAGE columns not found for dropping! "
    f"This means they either don't exist (schema change) or were renamed: {leakage_missing}. "
    f"Actual columns containing 'denial': {[c for c in df.columns if 'denial' in c]}"
)
print("\n✓ All 12 leakage columns confirmed present and will be dropped.")

Total columns in DataFrame:  99
Columns to drop:             58
  Leakage:              12
  Identifiers:          5
  Near-zero variance:   8
  Demographic detail:   28
  AUS detail:           4
  Target source:        1
Columns remaining:           41

✓ All 12 leakage columns confirmed present and will be dropped.


## Section 3 — Target Variable Creation & Population Filtering

**WHY filter BEFORE feature engineering?**

We only want `action_taken ∈ {1, 3}` (originated or denied). Other values (withdrawn, incomplete, purchased) represent fundamentally different populations. If we engineer features on the full dataset and then filter:
1. **Wasted compute** — transforming rows we'll discard
2. **Skewed statistics** — median income, missing rates, etc. get distorted by irrelevant records
3. **Wrong calibration** — features are tuned to a different distribution than what the model sees

**WHY extract `high_cost_flag` from `hoepa_status` BEFORE dropping it?**

`hoepa_status` appears in our NZV list (>99% value "2"). But EDA Cell 14 showed that the rare "1" (high-cost loan under HOEPA) correlates with dramatically higher denial rates. Converting to binary preserves the signal while acknowledging the NZV issue — the best of both worlds.

**WHY `label = 1` for denied (minority class)?**

Convention in binary classification: the minority/event class is the "positive" class (label=1). This aligns with how precision, recall, and F1 are calculated — they all focus on the positive class. Since denial is the event we want to detect, it gets label=1.

In [5]:
# ============================================================
# Cell 4: Filter to Target Population & Create Label
# ============================================================

start = time.time()

# STEP 1: Extract high_cost_flag BEFORE dropping hoepa_status
# WHY? hoepa_status is NZV (>99% "2"), but the rare "1" is an
# extremely strong signal. We convert to a clean binary flag.
df = df.withColumn(
    "high_cost_flag",
    F.when(F.col("hoepa_status").cast("string") == "1", 1.0).otherwise(0.0)
)

# STEP 2: Filter to originated (1) and denied (3) ONLY
# WHY before dropping columns? Row reduction first = everything after is faster
df_filtered = df.filter(F.col("action_taken").isin(1, 3))

# STEP 3: Create binary label
df_filtered = df_filtered.withColumn(
    "label",
    F.when(F.col("action_taken") == 3, 1.0).otherwise(0.0)
)

# STEP 4: Drop all identified columns
df_eng = df_filtered.drop(*existing_drops)

# ── 🐛 FIX (v2): Post-drop leakage verification ──
# Belt-and-suspenders: even after dropping, verify NO leakage columns remain.
LEAKAGE_PATTERNS = ["denial_reason", "rate_spread", "purchaser_type",
                    "total_loan_costs", "total_points_and_fees",
                    "origination_charges", "discount_points",
                    "lender_credits", "prepayment_penalty"]
surviving_leakage = [c for c in df_eng.columns
                     for pat in LEAKAGE_PATTERNS if pat in c]
assert len(surviving_leakage) == 0, (
    f"LEAKAGE DETECTED: {len(surviving_leakage)} columns with leakage "
    f"patterns survived the drop: {surviving_leakage}"
)

# ── Verify ──
target_total    = df_eng.count()
denied_count    = df_eng.filter(F.col("label") == 1).count()
originated_count = target_total - denied_count

print(f"✓ Feature engineering DataFrame created in {time.time()-start:.1f}s")
print(f"  Total records:        {target_total:,}")
print(f"  Originated (label=0): {originated_count:,} ({originated_count/target_total*100:.1f}%)")
print(f"  Denied     (label=1): {denied_count:,}   ({denied_count/target_total*100:.1f}%)")
print(f"  Columns remaining:    {len(df_eng.columns)}")
print(f"  Imbalance ratio:      {originated_count/max(denied_count,1):.1f}:1")
print(f"  ✓ Post-drop leakage check PASSED (0 leakage columns remaining)")

# Cache — we'll transform this DataFrame many times
df_eng.cache()
df_eng.count()  # Force materialization
print("✓ DataFrame cached.")

# List remaining columns
print(f"\nRemaining {len(df_eng.columns)} columns:")
for i, c in enumerate(sorted(df_eng.columns), 1):
    print(f"  {i:2d}. {c}")

✓ Feature engineering DataFrame created in 1.3s
  Total records:        7,686,484
  Originated (label=0): 5,691,726 (74.0%)
  Denied     (label=1): 1,994,758   (26.0%)
  Columns remaining:    43
  Imbalance ratio:      2.9:1
  ✓ Post-drop leakage check PASSED (0 leakage columns remaining)


26/03/01 17:39:50 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


✓ DataFrame cached.

Remaining 43 columns:
   1. applicant_age
   2. applicant_age_above_62
   3. applicant_credit_score_type
   4. aus_1
   5. co_applicant_age
   6. co_applicant_age_above_62
   7. co_applicant_credit_score_type
   8. combined_loan_to_value_ratio
   9. conforming_loan_limit
  10. construction_method
  11. debt_to_income_ratio
  12. derived_dwelling_category
  13. derived_ethnicity
  14. derived_loan_product_type
  15. derived_race
  16. derived_sex
  17. ffiec_msa_md_median_family_income
  18. high_cost_flag
  19. income
  20. initially_payable_to_institution
  21. interest_rate
  22. intro_rate_period
  23. label
  24. lien_status
  25. loan_amount
  26. loan_purpose
  27. loan_term
  28. loan_type
  29. manufactured_home_land_property_interest
  30. manufactured_home_secured_property_type
  31. multifamily_affordable_units
  32. occupancy_type
  33. preapproval
  34. property_value
  35. state_code
  36. submission_of_application
  37. total_units
  38. tract_median

## Section 4 — Missing Value Normalization: HMDA's Triple-Missing Problem

**WHY is this the hardest part of HMDA feature engineering?**

HMDA data has **three types** of "missing" that look different to Spark but all mean "no usable data":

| What Spark sees | What it means | Example | Our action |
|:---|:---|:---|:---|
| `null` | Truly empty cell | — | Leave as null |
| `"Exempt"` | Lender too small to report | Small bank that files <500 applications | → null |
| `"NA"` / `"Not applicable"` | Field doesn't apply to this loan type | Manufactured home field for a condo | → null |
| `"1111"` | Exempt code in numeric fields | Same as "Exempt" but for integer columns | → null |

**WHY convert ALL to null first, THEN impute?**

PySpark's `Imputer` only recognizes actual `null` values. If you leave `"Exempt"` as a string in a column you want to treat as numeric, the Imputer skips it — and then `VectorAssembler` crashes when it finds a string in what should be a numeric vector.

**Imputation strategy (from EDA analysis):**
- **Numeric → Median** — robust to the heavy right skew found in EDA Cell 4 (mean is pulled by outliers; median isn't)
- **Categorical → "Unknown"** — explicit placeholder that StringIndexer handles naturally

In [6]:
# ============================================================
# Cell 5: Normalize Missing Values → null
# ============================================================

# WHY a separate step (not inside the Pipeline)?
# 1. Imputer requires null, not string values
# 2. String-to-null conversion is domain-specific (HMDA), not a generic ML step
# 3. Must happen before we cast columns to DoubleType
# 4. Doing it before Pipeline keeps Pipeline stages generic & reusable

MISSING_VALUES = ["Exempt", "NA", "N/A", "Not applicable", "1111", ""]

print("Normalizing HMDA missing indicators → null ...")
for col_name in df_eng.columns:
    if col_name == "label":
        continue  # NEVER touch the target

    df_eng = df_eng.withColumn(
        col_name,
        F.when(
            F.col(col_name).cast("string").isin(MISSING_VALUES),
            F.lit(None)
        ).otherwise(F.col(col_name))
    )

print("✓ Converted Exempt/NA/1111/'' → null for all non-target columns")

# ── Show top missing columns ──
null_counts = df_eng.select([
    F.count(F.when(F.col(c).isNull(), True)).alias(c)
    for c in df_eng.columns if c != "label"
])
null_dict = null_counts.collect()[0].asDict()

print(f"\n{'Column':<45} {'Nulls':>12} {'% Missing':>10}")
print("-" * 70)
for col_name in sorted(null_dict, key=lambda x: -null_dict[x]):
    nulls = null_dict[col_name]
    pct = nulls / target_total * 100
    if pct > 1:
        flag = "🔴" if pct > 50 else "🟡" if pct > 20 else "  "
        print(f"{flag} {col_name:<43} {nulls:>12,} {pct:>9.1f}%")
print("\n🔴 >50% missing | 🟡 >20% missing")

Normalizing HMDA missing indicators → null ...
✓ Converted Exempt/NA/1111/'' → null for all non-target columns



Column                                               Nulls  % Missing
----------------------------------------------------------------------
🔴 multifamily_affordable_units                   7,661,839      99.7%
🔴 intro_rate_period                              5,754,560      74.9%
🔴 co_applicant_age_above_62                      4,499,227      58.5%
🟡 interest_rate                                  2,211,322      28.8%
   combined_loan_to_value_ratio                     751,069       9.8%
   debt_to_income_ratio                             627,872       8.2%
   property_value                                   436,910       5.7%
   loan_term                                        381,964       5.0%
   income                                           345,895       4.5%
   aus_1                                            234,731       3.1%
   submission_of_application                        232,176       3.0%
   initially_payable_to_institution                 232,176       3.0%
   applica

In [7]:
# ============================================================
# Cell 6: Create Missingness Indicator Features
# ============================================================

# WHY create "is_missing" features?
# EDA Cell 20 showed that for some columns, MISSINGNESS ITSELF predicts denial.
# Example: interest_rate is missing for ~40% of denied but ~10% of originated
# → "is interest_rate missing?" is a free, powerful signal.
#
# WHICH columns get indicators?
# Only those where EDA found >2% gap in denial rate between present & missing.
# Creating indicators for EVERY column would add too much noise.

INFORMATIVE_MISSING_COLS = [
    "interest_rate",                    # Often missing for denied (no rate offered)
    "combined_loan_to_value_ratio",     # Missing when property value unknown
    "property_value",                   # Missing more for denied
    "income",                           # Missing more for denied
    "debt_to_income_ratio",             # String field, often "Exempt"
    "loan_term",                        # Missing for some denied apps
    "intro_rate_period",                # Mostly missing — but informative when present
    "applicant_credit_score_type",      # Credit score type varies by denial outcome
    "co_applicant_credit_score_type",   # Co-applicant presence is itself a signal
]

indicator_cols_created = []
for col_name in INFORMATIVE_MISSING_COLS:
    if col_name in df_eng.columns:
        ind_name = f"{col_name}_is_missing"
        df_eng = df_eng.withColumn(
            ind_name,
            F.when(F.col(col_name).isNull(), 1.0).otherwise(0.0)
        )
        indicator_cols_created.append(ind_name)

print(f"✓ Created {len(indicator_cols_created)} missingness indicator features:")
for c in indicator_cols_created:
    ct = df_eng.filter(F.col(c) == 1).count()
    print(f"  {c:<50} {ct:>10,} ({ct/target_total*100:.1f}%)")

✓ Created 9 missingness indicator features:
  interest_rate_is_missing                            2,211,322 (28.8%)
  combined_loan_to_value_ratio_is_missing               751,069 (9.8%)
  property_value_is_missing                             436,910 (5.7%)
  income_is_missing                                     345,895 (4.5%)
  debt_to_income_ratio_is_missing                       627,872 (8.2%)
  loan_term_is_missing                                  381,964 (5.0%)
  intro_rate_period_is_missing                        5,754,560 (74.9%)
  applicant_credit_score_type_is_missing                232,009 (3.0%)
  co_applicant_credit_score_type_is_missing             232,009 (3.0%)


## Section 5 — Numeric Feature Transformations

**WHY transform numeric features?**

Raw numeric values have three problems that hurt ML models:

### Problem 1: Skewness (EDA Cell 4-5)
`loan_amount`, `income`, `property_value` are heavily right-skewed (skewness >> 1).
- **Impact on models:** Linear models (Logistic Regression) assume roughly symmetric features. A \$10M loan is 100× larger than \$100K, but shouldn't have 100× the influence on the decision boundary.
- **Fix:** `log1p(x)` compresses the right tail. `log1p(100000) ≈ 11.5`, `log1p(10000000) ≈ 16.1` — now only ~1.4× different instead of 100×.

### Problem 2: Outliers (EDA Cell 6-7)
`combined_loan_to_value_ratio` has values like 999 (likely "Exempt" codes that survived casting).
- **Impact:** Outliers dominate gradient updates in Logistic Regression and create tiny, overfit tree splits.
- **Fix:** Clip to domain-reasonable bounds (CLTV capped at 200%).

### Problem 3: Missing Feature Interactions
Raw columns miss domain-critical combinations:
- **`loan_to_income_ratio`**: A \$500K loan is fine with \$200K income (ratio=2.5) but disastrous with \$30K income (ratio=16.7). This single feature captures *affordability* — the #1 underwriting factor.
- **`is_joint_application`**: Two incomes vs one fundamentally changes the risk profile.

**CRITICAL:** All transformations happen BEFORE the Pipeline's StandardScaler. The scaler expects clean, numeric input — not strings or uncapped outliers.

In [8]:
# ============================================================
# Cell 7: Cast Numeric Columns to DoubleType
# ============================================================

# WHY this step?
# After replacing "Exempt"/"NA" with null in Cell 5, the columns
# are STILL StringType in Spark's schema (one string value makes
# the whole column StringType). We must explicitly cast to Double
# so that Imputer and VectorAssembler can process them.
#
# Without this: "Column X must be numeric type but was StringType"

# Which columns should be numeric?
NUMERIC_FEATURES = [c for c in NUMERIC_CONTINUOUS
                    if c in df_eng.columns and c not in LEAKAGE_DROP]

# Plus our engineered numeric features
ADDITIONAL_NUMERIC = ["high_cost_flag"]
ALL_NUMERIC = NUMERIC_FEATURES + ADDITIONAL_NUMERIC

print("Casting columns to DoubleType ...")
cast_count = 0
for col_name in ALL_NUMERIC:
    if col_name not in df_eng.columns:
        continue
    current = str(df_eng.schema[col_name].dataType)
    if current not in ("DoubleType",):
        df_eng = df_eng.withColumn(col_name, F.col(col_name).cast("double"))
        cast_count += 1

# Also ensure indicator columns are Double (they should already be)
for col_name in indicator_cols_created:
    df_eng = df_eng.withColumn(col_name, F.col(col_name).cast("double"))

print(f"✓ Cast {cast_count} columns to DoubleType")
print(f"✓ Total numeric features: {len(ALL_NUMERIC)}")

Casting columns to DoubleType ...
✓ Cast 16 columns to DoubleType
✓ Total numeric features: 16


In [9]:
# ============================================================
# Cell 8: Log Transform Skewed Numeric Features
# ============================================================

# WHY log transform?
# EDA Cell 4 showed loan_amount, income, property_value have
# skewness > 2. Log transform:
# 1. Compresses right tail → reduces outlier impact
# 2. Makes distribution more symmetric → linear models work better
# 3. Is monotonic → preserves ranking (larger income stays larger)
#
# WHY log1p (log(1+x)) instead of log(x)?
# log(0) = -infinity. Some records have income=0 or property_value=0.
# log1p(0) = 0, which is safe and meaningful.
#
# WHY create NEW columns (not overwrite)?
# We might want original values for interpretability later.
# The Pipeline's VectorAssembler will use the _log versions.

LOG_TRANSFORM_COLS = [
    "loan_amount",      # Right-skewed financial amount
    "income",           # Right-skewed income distribution
    "property_value",   # Right-skewed property values
    "tract_population", # Population varies hugely (rural vs urban)
]

print("=" * 80)
print("LOG TRANSFORMATIONS: log1p(x)")
print("=" * 80)
print(f"  {'Column':<30} {'Mean':>12} → {'Log Mean':>12}  {'Max':>14} → {'Log Max':>12}")
print("-" * 90)

log_cols_created = []
for col_name in LOG_TRANSFORM_COLS:
    if col_name not in df_eng.columns:
        continue

    new_col = f"{col_name}_log"
    # log1p handles null → null (safe propagation in Spark)
    df_eng = df_eng.withColumn(
        new_col,
        F.when(
            F.col(col_name).isNotNull() & (F.col(col_name) >= 0),
            F.log1p(F.col(col_name))
        ).otherwise(F.lit(None))
    )
    log_cols_created.append(new_col)

    # Report transformation effect
    s = df_eng.select(
        F.mean(col_name).alias("om"), F.max(col_name).alias("ox"),
        F.mean(new_col).alias("lm"), F.max(new_col).alias("lx")
    ).first()

    print(f"  {col_name:<30} {s['om'] or 0:>12.1f} → {s['lm'] or 0:>12.3f}  {s['ox'] or 0:>14.1f} → {s['lx'] or 0:>12.3f}")

print(f"\n✓ Created {len(log_cols_created)} log-transformed columns")
print("  Original columns kept for interpretability. Pipeline uses _log versions.")

LOG TRANSFORMATIONS: log1p(x)
  Column                                 Mean →     Log Mean             Max →      Log Max
------------------------------------------------------------------------------------------
  loan_amount                        313375.0 →       12.026  232323235000.0 →       26.171
  income                                183.8 →        4.617     132000000.0 →       18.698
  property_value                     534768.9 →       12.818    2147483647.0 →       21.488
  tract_population                     4760.3 →        8.284         30199.0 →       10.316

✓ Created 4 log-transformed columns
  Original columns kept for interpretability. Pipeline uses _log versions.


In [10]:
# ============================================================
# Cell 9: Clip Outliers & Create Engineered Features
# ============================================================

# ──── A. CLIP OUTLIERS ────
# WHY clip instead of remove? Removing outlier ROWS loses all their
# other feature values. Clipping caps the extreme value, preserving the row.

print("=" * 80)
print("OUTLIER CLIPPING & FEATURE ENGINEERING")
print("=" * 80)

# combined_loan_to_value_ratio: Valid range [0, 200%]
# Values > 200% are either data errors or "Exempt" codes surviving as numbers
if "combined_loan_to_value_ratio" in df_eng.columns:
    ct = df_eng.filter(F.col("combined_loan_to_value_ratio") > 200).count()
    df_eng = df_eng.withColumn(
        "combined_loan_to_value_ratio",
        F.when(F.col("combined_loan_to_value_ratio") > 200, 200.0)
        .when(F.col("combined_loan_to_value_ratio") < 0, F.lit(None))
        .otherwise(F.col("combined_loan_to_value_ratio"))
    )
    print(f"  CLTV: Clipped {ct:,} rows > 200% to 200")

# interest_rate: Valid range [0, 30%] — no real mortgage exceeds this
if "interest_rate" in df_eng.columns:
    ct = df_eng.filter(F.col("interest_rate") > 30).count()
    df_eng = df_eng.withColumn(
        "interest_rate",
        F.when(F.col("interest_rate") > 30, F.lit(None))
        .when(F.col("interest_rate") < 0, F.lit(None))
        .otherwise(F.col("interest_rate"))
    )
    print(f"  Interest rate: Nulled {ct:,} rows > 30% (likely errors)")

# ──── B. ENGINEERED FEATURES ────
# WHY? Domain knowledge creates features more predictive than raw amounts.

# B1: loan_to_income_ratio — CORE AFFORDABILITY METRIC
# Every underwriter looks at this. $500K loan / $200K income = 2.5 (safe)
# $500K loan / $30K income = 16.7 (very risky)
if "loan_amount" in df_eng.columns and "income" in df_eng.columns:
    df_eng = df_eng.withColumn(
        "loan_to_income_ratio",
        F.when(
            F.col("income").isNotNull() & (F.col("income") > 0),
            F.col("loan_amount") / F.col("income")
        ).otherwise(F.lit(None))
    )
    # Clip extreme ratios (>50 is unrealistic for any mortgage product)
    df_eng = df_eng.withColumn(
        "loan_to_income_ratio",
        F.when(F.col("loan_to_income_ratio") > 50, 50.0)
        .otherwise(F.col("loan_to_income_ratio"))
    )
    # Also log-transform this ratio (it's skewed too)
    df_eng = df_eng.withColumn(
        "loan_to_income_ratio_log",
        F.when(F.col("loan_to_income_ratio").isNotNull() & (F.col("loan_to_income_ratio") > 0),
               F.log1p(F.col("loan_to_income_ratio")))
        .otherwise(F.lit(None))
    )
    print("  ✓ Created loan_to_income_ratio + loan_to_income_ratio_log")

# B2: is_joint_application — fundamentally changes risk profile
if "co_applicant_age" in df_eng.columns:
    df_eng = df_eng.withColumn(
        "is_joint_application",
        F.when(
            F.col("co_applicant_age").isNotNull() &
            ~F.col("co_applicant_age").cast("string").isin("9999", "8888"),
            1.0
        ).otherwise(0.0)
    )
    print("  ✓ Created is_joint_application (from co_applicant_age)")

# B3: Cast remaining numeric-looking columns
for c in ["tract_to_msa_income_percentage", "tract_minority_population_percent",
           "ffiec_msa_md_median_family_income", "tract_owner_occupied_units",
           "tract_one_to_four_family_homes", "tract_median_age_of_housing_units"]:
    if c in df_eng.columns:
        df_eng = df_eng.withColumn(c, F.col(c).cast("double"))

print(f"\n✓ DataFrame now has {len(df_eng.columns)} columns (including engineered features)")

OUTLIER CLIPPING & FEATURE ENGINEERING
  CLTV: Clipped 4,890 rows > 200% to 200
  Interest rate: Nulled 1 rows > 30% (likely errors)
  ✓ Created loan_to_income_ratio + loan_to_income_ratio_log
  ✓ Created is_joint_application (from co_applicant_age)

✓ DataFrame now has 59 columns (including engineered features)


## Section 6 — Categorical Encoding Strategy

**WHY can't ML models use strings?**
Every ML algorithm operates on numeric vectors. The string `"Conventional"` is mathematically meaningless. We must convert — but **how** matters enormously.

### Three strategies (chosen by cardinality from EDA Cell 8):

| Strategy | When | How | Example |
|:---|:---|:---|:---|
| **StringIndexer + OneHot** | 2-10 unique values | Creates binary columns | `loan_type` → [1,0,0,0], [0,1,0,0], ... |
| **StringIndexer only** | 11-50 unique values | Maps to integer index | `debt_to_income_ratio` → 0, 1, 2, ... |
| **DROP** | 50+ unique values | Too many to encode | `state_code` (51 values) → use census features instead |

### Why `handleInvalid="keep"` everywhere?
In production, test data may contain categories not seen during training. `"keep"` maps unseen values to a special index instead of crashing. `"error"` would crash; `"skip"` would silently lose rows — both dangerous.

### Why `dropLast=True` in OneHotEncoder?
Avoids the **dummy variable trap**: if a feature has values {A, B, C}, we only need 2 columns (A=[1,0], B=[0,1], C=[0,0]). The third is perfectly correlated with the first two → causes multicollinearity in linear models.

In [11]:
# ============================================================
# Cell 10: Classify Categoricals by Cardinality
# ============================================================

# Identify remaining categorical columns
# (everything that's not numeric, not label, not our engineered features)
ENGINEERED = (log_cols_created + indicator_cols_created +
              ["loan_to_income_ratio", "loan_to_income_ratio_log",
               "is_joint_application", "high_cost_flag"])

remaining_cats = [c for c in df_eng.columns
                  if c not in ALL_NUMERIC
                  and c not in ENGINEERED
                  and c != "label"]

print("=" * 80)
print("CATEGORICAL ENCODING PLAN")
print("=" * 80)
print(f"  {'Column':<43} {'Unique':>8} {'Strategy':<25}")
print("-" * 80)

CAT_ONEHOT = []       # 2-10 values → StringIndexer + OneHotEncoder
CAT_INDEX_ONLY = []   # 11-50 values → StringIndexer only
CAT_DROP_HIGH = []    # 50+ values → DROP (too many)

for col_name in sorted(remaining_cats):
    n_unique = df_eng.select(col_name).distinct().count()

    if n_unique <= 1:
        strategy = "DROP (zero variance)"
        CAT_DROP_HIGH.append(col_name)
    elif n_unique <= 10:
        strategy = "OneHot"
        CAT_ONEHOT.append(col_name)
    elif n_unique <= 50:
        strategy = "IndexOnly"
        CAT_INDEX_ONLY.append(col_name)
    else:
        strategy = "DROP (>50 unique)"
        CAT_DROP_HIGH.append(col_name)

    print(f"  {col_name:<43} {n_unique:>8} {strategy}")

print(f"\nSummary:")
print(f"  OneHot encoding:     {len(CAT_ONEHOT)} columns")
print(f"  Index-only encoding: {len(CAT_INDEX_ONLY)} columns")
print(f"  Drop (high card.):   {len(CAT_DROP_HIGH)} → {CAT_DROP_HIGH}")

if CAT_DROP_HIGH:
    df_eng = df_eng.drop(*CAT_DROP_HIGH)
    print(f"\n✓ Dropped {len(CAT_DROP_HIGH)} high-cardinality columns")

CATEGORICAL ENCODING PLAN
  Column                                        Unique Strategy                 
--------------------------------------------------------------------------------
  applicant_age                                      8 OneHot
  applicant_age_above_62                             3 OneHot
  applicant_credit_score_type                       11 IndexOnly
  aus_1                                              8 OneHot
  co_applicant_age                                   9 OneHot
  co_applicant_age_above_62                          3 OneHot
  co_applicant_credit_score_type                    12 IndexOnly
  conforming_loan_limit                              4 OneHot
  construction_method                                2 OneHot
  debt_to_income_ratio                              20 IndexOnly
  derived_dwelling_category                          4 OneHot
  derived_ethnicity                                  5 OneHot
  derived_loan_product_type                          8 OneH

In [12]:
# ============================================================
# Cell 11: Prepare Categoricals for StringIndexer
# ============================================================

# StringIndexer REQUIRES StringType input. Some "categorical" columns
# were read as IntegerType (e.g., loan_type = 1,2,3,4 — integers,
# but they're category codes, not quantities). Cast to String.

ALL_CATEGORICALS = CAT_ONEHOT + CAT_INDEX_ONLY

cast_ct = 0
for col_name in ALL_CATEGORICALS:
    if col_name in df_eng.columns:
        if str(df_eng.schema[col_name].dataType) != "StringType":
            df_eng = df_eng.withColumn(col_name, F.col(col_name).cast("string"))
            cast_ct += 1

# Fill null categoricals with "Unknown"
# WHY? StringIndexer CAN handle nulls, but explicit "Unknown" is cleaner
# and makes the category visible in feature importance analysis later.
for col_name in ALL_CATEGORICALS:
    if col_name in df_eng.columns:
        df_eng = df_eng.withColumn(
            col_name,
            F.when(F.col(col_name).isNull(), "Unknown").otherwise(F.col(col_name))
        )

print(f"✓ Cast {cast_ct} categorical columns to StringType")
print(f"✓ Filled null categoricals with 'Unknown'")
print(f"✓ Total categoricals for encoding: {len(ALL_CATEGORICALS)}")

✓ Cast 25 categorical columns to StringType
✓ Filled null categoricals with 'Unknown'
✓ Total categoricals for encoding: 25


## Section 7 — PySpark ML Pipeline Assembly

**This is the architectural centerpiece of the notebook.**

### Why Pipeline over manual transforms?

| Manual Approach | Pipeline Approach |
|:---|:---|
| Risk of data leakage (fit on all data) | `fit(train)` → `transform(test)` automatically |
| Can't serialize for production | `pipeline_model.save()` / `load()` |
| Must repeat steps for each new dataset | Single `transform()` handles everything |
| Easy to forget/reorder steps | Fixed, auditable sequence |

### Our Pipeline stages, in order:

```
Stage 1: Imputer (numeric nulls → median)
    ↓  WHY first? VectorAssembler crashes on nulls.
Stage 2: StringIndexer × N (categorical strings → numeric indices)
    ↓  WHY before OHE? OneHotEncoder needs numeric input.
Stage 3: OneHotEncoder × M (low-cardinality indices → sparse binary vectors)
    ↓  WHY only low-card? 50-category OneHot = 49 columns → curse of dimensionality.
Stage 4: VectorAssembler (all features → single "features" vector)
    ↓  WHY? PySpark ML models require one vector column.
Stage 5: StandardScaler (zero mean / unit variance)
    ↓  WHY last? Must scale the assembled vector, not individual columns.
```

### Critical: `withMean=False` in StandardScaler
OneHotEncoder creates **sparse** vectors (mostly zeros, stored efficiently). If we center them (`withMean=True`), they become **dense** — every zero becomes a small number, and memory usage explodes 10-100×. We scale variance only (`withStd=True`).

In [13]:
# ============================================================
# Cell 12: Build Pipeline Stages
# ============================================================

start = time.time()

# ──── STAGE 1: Numeric Imputation ────
# Collect all numeric (Double/Float) columns, excluding label and categoricals.
#
# BUG FIX: str(DoubleType()) returns "double", NOT "DoubleType".
# We use isinstance() which is the robust, Pythonic way to check types.

from pyspark.sql.types import DoubleType, FloatType

numeric_to_impute = []
for col_name in df_eng.columns:
    if col_name == "label" or col_name in ALL_CATEGORICALS:
        continue
    col_type = df_eng.schema[col_name].dataType
    if isinstance(col_type, (DoubleType, FloatType)):
        numeric_to_impute.append(col_name)

print(f"Found {len(numeric_to_impute)} numeric columns for imputation:")
for c in numeric_to_impute:
    print(f"  {c}")

# Safety check — Imputer crashes with empty inputCols
assert len(numeric_to_impute) > 0, \
    f"No numeric columns found! Check Cell 7 casting. " \
    f"Column dtypes: {[(c, str(df_eng.schema[c].dataType)) for c in df_eng.columns[:10]]}"

imputer_output = [f"{c}_imp" for c in numeric_to_impute]

imputer = Imputer(
    inputCols=numeric_to_impute,
    outputCols=imputer_output,
    strategy="median"  # Robust to skew — better than mean for financial data
)
print(f"\nStage 1: Imputer — {len(numeric_to_impute)} numeric columns → median")

# ──── STAGE 2: StringIndexer for ALL categoricals ────
indexers = []
indexed_col_names = []

for col_name in ALL_CATEGORICALS:
    if col_name not in df_eng.columns:
        continue
    out = f"{col_name}_idx"
    indexers.append(StringIndexer(
        inputCol=col_name, outputCol=out,
        handleInvalid="keep"  # Unseen categories → special index (won't crash)
    ))
    indexed_col_names.append(out)

print(f"Stage 2: StringIndexer × {len(indexers)}")

# ──── STAGE 3: OneHotEncoder for LOW cardinality only ────
ohe_in  = [f"{c}_idx" for c in CAT_ONEHOT if c in df_eng.columns]
ohe_out = [f"{c}_ohe" for c in CAT_ONEHOT if c in df_eng.columns]

encoder = OneHotEncoder(
    inputCols=ohe_in, outputCols=ohe_out,
    dropLast=True  # Avoids dummy variable trap
)
print(f"Stage 3: OneHotEncoder × {len(ohe_in)}")

# ──── STAGE 4: VectorAssembler ────
# Collect ALL feature columns for the final vector
feature_cols = []
feature_cols.extend(imputer_output)                                            # Imputed numerics
feature_cols.extend(ohe_out)                                                    # OneHot vectors
feature_cols.extend([f"{c}_idx" for c in CAT_INDEX_ONLY if c in df_eng.columns])  # Index-only cats

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features_unscaled",
    handleInvalid="skip"  # Drop rows with ANY remaining null
)
print(f"Stage 4: VectorAssembler — {len(feature_cols)} inputs → 'features_unscaled'")

# ──── STAGE 5: StandardScaler ────
scaler = StandardScaler(
    inputCol="features_unscaled",
    outputCol="features",
    withMean=False,  # CRITICAL: sparse + withMean=True = dense = OOM
    withStd=True     # Normalize to unit variance
)
print(f"Stage 5: StandardScaler → 'features'")

# ──── ASSEMBLE PIPELINE ────
pipeline = Pipeline(stages=[imputer] + indexers + [encoder, assembler, scaler])

print(f"\n{'='*60}")
print(f"PIPELINE ASSEMBLED: {len(pipeline.getStages())} total stages")
print(f"{'='*60}")
print(f"  Imputed numerics:       {len(imputer_output)}")
print(f"  OneHot encoded:         {len(ohe_out)}")
print(f"  Index-only cats:        {len([f'{c}_idx' for c in CAT_INDEX_ONLY if c in df_eng.columns])}")
print(f"  Total assembler inputs: {len(feature_cols)}")
print(f"  Construction time:      {time.time()-start:.1f}s")

Found 32 numeric columns for imputation:
  loan_amount
  combined_loan_to_value_ratio
  interest_rate
  loan_term
  intro_rate_period
  property_value
  multifamily_affordable_units
  income
  tract_population
  tract_minority_population_percent
  ffiec_msa_md_median_family_income
  tract_to_msa_income_percentage
  tract_owner_occupied_units
  tract_one_to_four_family_homes
  tract_median_age_of_housing_units
  high_cost_flag
  interest_rate_is_missing
  combined_loan_to_value_ratio_is_missing
  property_value_is_missing
  income_is_missing
  debt_to_income_ratio_is_missing
  loan_term_is_missing
  intro_rate_period_is_missing
  applicant_credit_score_type_is_missing
  co_applicant_credit_score_type_is_missing
  loan_amount_log
  income_log
  property_value_log
  tract_population_log
  loan_to_income_ratio
  loan_to_income_ratio_log
  is_joint_application

Stage 1: Imputer — 32 numeric columns → median
Stage 2: StringIndexer × 25
Stage 3: OneHotEncoder × 22
Stage 4: VectorAssembler — 5

## Section 8 — Stratified Train/Test Split & Pipeline Fitting

### Why split BEFORE fitting the Pipeline?

This is the **single most critical step** for preventing data leakage.

If we fit the Pipeline on ALL data and then split:
- StringIndexer learns category distributions from test data → **leakage**
- Imputer computes medians using test data values → **leakage**
- StandardScaler's std includes test data → **leakage**

**Correct workflow:**
```
1. Split: df → train (80%) + test (20%)
2. Pipeline.fit(train)  → learns all parameters from TRAIN ONLY
3. PipelineModel.transform(train) → training features
4. PipelineModel.transform(test)  → test features (same parameters!)
```

### Why stratified split?
Our target is imbalanced (~80% originated, ~20% denied). A random split could produce:
- Train: 82% / 18%
- Test: 75% / 25%

→ Different populations → misleading metrics. Stratified split ensures both sets have the same class ratio.

### Why 80/20?
With ~8M rows, even 20% test = ~1.6M records — more than enough for reliable evaluation. 70/30 wastes training data; 90/10 gives too few test examples.

In [14]:
# ============================================================
# Cell 13: Stratified Train/Test Split
# ============================================================

start = time.time()

# WHY sampleBy instead of randomSplit?
# randomSplit doesn't guarantee class proportions.
# sampleBy lets us specify the EXACT fraction per class.

TRAIN_FRACTION = 0.8
SEED = 42  # Fixed for reproducibility

train_df = df_eng.sampleBy("label", fractions={0.0: TRAIN_FRACTION, 1.0: TRAIN_FRACTION}, seed=SEED)
test_df  = df_eng.subtract(train_df)  # Complement

# Verify
train_total   = train_df.count()
train_denied  = train_df.filter(F.col("label") == 1).count()
train_orig    = train_total - train_denied

test_total    = test_df.count()
test_denied   = test_df.filter(F.col("label") == 1).count()
test_orig     = test_total - test_denied

print("=" * 70)
print("STRATIFIED TRAIN/TEST SPLIT")
print("=" * 70)
print(f"  {'Set':<16} {'Total':>12} {'Originated':>12} {'Denied':>12} {'Denial %':>10}")
print("-" * 65)
print(f"  {'Train (80%)':<16} {train_total:>12,} {train_orig:>12,} {train_denied:>12,} {train_denied/train_total*100:>9.1f}%")
print(f"  {'Test  (20%)':<16} {test_total:>12,} {test_orig:>12,} {test_denied:>12,} {test_denied/test_total*100:>9.1f}%")
print(f"  {'Full':<16} {target_total:>12,} {originated_count:>12,} {denied_count:>12,} {denied_count/target_total*100:>9.1f}%")
print(f"\n✓ Split in {time.time()-start:.1f}s")
print(f"  Denial rate preserved: train={train_denied/train_total*100:.2f}% vs test={test_denied/test_total*100:.2f}%")

STRATIFIED TRAIN/TEST SPLIT
  Set                     Total   Originated       Denied   Denial %
-----------------------------------------------------------------
  Train (80%)         6,148,713    4,552,842    1,595,871      26.0%
  Test  (20%)         1,533,840    1,136,777      397,063      25.9%
  Full                7,686,484    5,691,726    1,994,758      26.0%

✓ Split in 192.4s
  Denial rate preserved: train=25.95% vs test=25.89%


In [15]:
# ============================================================
# Cell 14: Fit Pipeline on TRAINING Data Only — Then Transform Both
# ============================================================

# THIS IS THE KEY DATA LEAKAGE PREVENTION STEP.
#
# Pipeline.fit(train_df) learns:
#   - Imputer:        median values from training data
#   - StringIndexer:  category → index mappings from training data
#   - StandardScaler: std from training data
#
# PipelineModel.transform() applies these SAME learned parameters
# to both train and test — no information from test leaks into train.

start = time.time()

print("=" * 70)
print("FITTING PIPELINE ON TRAINING DATA")
print("=" * 70)
print(f"Training records:  {train_total:,}")
print(f"Pipeline stages:   {len(pipeline.getStages())}")
print("\nFitting... (may take 2-5 minutes depending on data size)")

pipeline_model = pipeline.fit(train_df)
fit_time = time.time() - start
print(f"\n✓ Pipeline fitted in {fit_time:.1f}s")

# ── Transform both sets ──
print("\nTransforming train set...")
t0 = time.time()
train_transformed = pipeline_model.transform(train_df)
print(f"  ✓ Done in {time.time()-t0:.1f}s")

print("Transforming test set...")
t0 = time.time()
test_transformed = pipeline_model.transform(test_df)
print(f"  ✓ Done in {time.time()-t0:.1f}s")

# Keep only what the model needs: label + features
train_final = train_transformed.select("label", "features")
test_final  = test_transformed.select("label", "features")

# Cache & materialize
train_final.cache()
test_final.cache()
train_count = train_final.count()
test_count  = test_final.count()

train_dim = train_final.first()['features'].size
print(f"\n✓ Final datasets ready:")
print(f"  Train: {train_count:,} rows")
print(f"  Test:  {test_count:,} rows")
print(f"  Feature vector size: {train_dim}")

FITTING PIPELINE ON TRAINING DATA
Training records:  6,148,713
Pipeline stages:   29

Fitting... (may take 2-5 minutes depending on data size)



✓ Pipeline fitted in 124.3s

Transforming train set...
  ✓ Done in 0.7s
Transforming test set...
  ✓ Done in 1.1s


26/03/01 17:47:46 WARN MemoryStore: Not enough space to cache rdd_769_2 in memory! (computed 452.1 MiB so far)
26/03/01 17:47:46 WARN BlockManager: Persisting block rdd_769_2 to disk instead.
26/03/01 17:47:47 WARN MemoryStore: Not enough space to cache rdd_769_3 in memory! (computed 452.1 MiB so far)
26/03/01 17:47:47 WARN BlockManager: Persisting block rdd_769_3 to disk instead.



✓ Final datasets ready:
  Train: 6,148,713 rows
  Test:  1,533,840 rows
  Feature vector size: 145


## Section 9 — Post-Pipeline Verification & Quality Checks

**"Trust but verify."** The Pipeline might have:
- Dropped too many rows (`handleInvalid="skip"` in VectorAssembler removes rows with ANY null)
- Created `NaN`/`Inf` in the feature vector
- Produced unexpected dimensions (off-by-one from OneHotEncoder's `dropLast`)
- Lost class balance during assembly

We run 5 checks to catch these issues before sending data to model training.

In [16]:
# ============================================================
# Cell 15: Post-Pipeline Quality Checks
# ============================================================

print("=" * 70)
print("POST-PIPELINE VERIFICATION")
print("=" * 70)

# ── CHECK 1: Row survival ──
processed = train_count + test_count
dropped_pct = (1 - processed / target_total) * 100
print(f"\n1. ROW COUNT")
print(f"   Original (filtered):  {target_total:,}")
print(f"   After pipeline:       {processed:,}")
print(f"   Rows dropped:         {target_total - processed:,} ({dropped_pct:.1f}%)")
if dropped_pct < 5:
    print("   ✓ Acceptable (< 5%)")
else:
    print("   ⚠ WARNING: >5% rows lost — investigate VectorAssembler nulls")

# ── CHECK 2: Feature dimensions ──
test_dim = test_final.first()['features'].size
print(f"\n2. FEATURE DIMENSIONS")
print(f"   Train: {train_dim}  |  Test: {test_dim}")
if train_dim == test_dim:
    print("   ✓ Match")
else:
    print("   ✗ MISMATCH — Pipeline bug!")

# ── CHECK 3: Class balance preservation ──
train_denial_rate = train_final.filter(F.col("label") == 1).count() / train_count
test_denial_rate  = test_final.filter(F.col("label") == 1).count() / test_count
print(f"\n3. CLASS BALANCE")
print(f"   Train denial rate: {train_denial_rate*100:.2f}%")
print(f"   Test denial rate:  {test_denial_rate*100:.2f}%")
print(f"   Difference:        {abs(train_denial_rate - test_denial_rate)*100:.2f}%")
if abs(train_denial_rate - test_denial_rate) < 0.5:
    print("   ✓ Preserved")
else:
    print("   ⚠ Drift detected")

# ── CHECK 4: Sample vectors ──
print(f"\n4. SAMPLE FEATURE VECTORS (first 5 rows, first 10 values)")
for row in train_final.take(5):
    vals = row['features'].toArray()[:10]
    vals_str = ", ".join([f"{v:.3f}" for v in vals])
    print(f"   label={int(row['label'])} | [{vals_str}, ...]")

# ── CHECK 5: Feature statistics ──
print(f"\n5. FEATURE STATISTICS")
summary = train_final.select(
    Summarizer.metrics("mean", "variance").summary(F.col("features"))
).collect()[0][0]

means = summary[0].toArray()
variances = summary[1].toArray()
zero_var = sum(1 for v in variances if v == 0)
print(f"   Total features:           {len(means)}")
print(f"   Zero-variance features:   {zero_var}")
print(f"   Mean range:  [{min(means):.4f}, {max(means):.4f}]")
print(f"   Var range:   [{min(variances):.4f}, {max(variances):.4f}]")
if zero_var == 0:
    print("   ✓ All features have non-zero variance")
else:
    print(f"   ⚠ {zero_var} zero-variance features found — consider removing")

POST-PIPELINE VERIFICATION

1. ROW COUNT
   Original (filtered):  7,686,484
   After pipeline:       7,682,553
   Rows dropped:         3,931 (0.1%)
   ✓ Acceptable (< 5%)

2. FEATURE DIMENSIONS
   Train: 145  |  Test: 145
   ✓ Match



3. CLASS BALANCE
   Train denial rate: 25.95%
   Test denial rate:  25.89%
   Difference:        0.07%
   ✓ Preserved

4. SAMPLE FEATURE VECTORS (first 5 rows, first 10 values)
   label=1 | [0.160, 4.906, 4.805, 4.236, 0.042, 0.067, 0.000, 0.002, 1.790, 3.181, ...]
   label=1 | [0.196, 3.215, 4.805, 4.236, 0.042, 0.124, 0.000, 0.005, 2.334, 1.913, ...]
   label=1 | [0.148, 3.100, 4.805, 4.236, 0.042, 0.097, 0.000, 0.007, 1.967, 2.351, ...]
   label=1 | [0.003, 4.528, 4.805, 0.424, 0.042, 0.075, 0.000, 0.002, 1.790, 3.181, ...]
   label=1 | [0.160, 4.585, 4.805, 4.236, 0.042, 0.075, 0.000, 0.003, 3.129, 3.238, ...]

5. FEATURE STATISTICS


   Total features:           145
   Zero-variance features:   0
   Mean range:  [0.0022, 145.7870]
   Var range:   [1.0000, 1.0000]
   ✓ All features have non-zero variance


In [17]:
# ============================================================
# Cell 16: Save Processed Data & Pipeline Model
# ============================================================

import os

# Save train/test Parquet
train_path = os.path.join(OUTPUT_DIR, "train_features.parquet")
test_path  = os.path.join(OUTPUT_DIR, "test_features.parquet")

train_final.write.mode("overwrite").parquet(train_path)
print(f"✓ Train saved: {train_path}")

test_final.write.mode("overwrite").parquet(test_path)
print(f"✓ Test saved:  {test_path}")

# Save pipeline model for production use
model_path = os.path.join(OUTPUT_DIR, "pipeline_model")
pipeline_model.write().overwrite().save(model_path)
print(f"✓ Pipeline model saved: {model_path}")

# Save feature metadata
feature_meta = {
    "n_features": train_dim,
    "n_train": train_count,
    "n_test": test_count,
    "train_denial_rate": float(train_denial_rate),
    "test_denial_rate": float(test_denial_rate),
    "numeric_features": numeric_to_impute,
    "onehot_categoricals": CAT_ONEHOT,
    "index_categoricals": CAT_INDEX_ONLY,
    "indicator_features": indicator_cols_created,
    "log_features": log_cols_created,
    "assembler_inputs": feature_cols,
}
meta_path = os.path.join(OUTPUT_DIR, "feature_metadata.json")
with open(meta_path, "w") as f:
    json.dump(feature_meta, f, indent=2)
print(f"✓ Feature metadata saved: {meta_path}")

✓ Train saved: /Users/adi/Desktop/assignmt/project/data/processed/train_features.parquet


26/03/01 17:50:54 WARN DAGScheduler: Broadcasting large task binary with size 1016.9 KiB


✓ Test saved:  /Users/adi/Desktop/assignmt/project/data/processed/test_features.parquet
✓ Pipeline model saved: /Users/adi/Desktop/assignmt/project/data/processed/pipeline_model
✓ Feature metadata saved: /Users/adi/Desktop/assignmt/project/data/processed/feature_metadata.json


In [18]:
# ============================================================
# Cell 17: Summary & Next Steps
# ============================================================

print("=" * 70)
print("NOTEBOOK 3 COMPLETE: Feature Engineering & Pipeline Construction")
print("=" * 70)

lines = [
    "",
    f"Started with {total_rows:,} rows x 99 columns",
    f"Normalized column names (hyphens → underscores)",
    f"Filtered to {target_total:,} rows (action_taken in {{1, 3}})",
    f"Dropped {len(existing_drops)} columns (leakage, IDs, NZV, redundant)",
    f"  ✓ Post-drop leakage assertion PASSED",
    f"Normalized HMDA missing values (Exempt/NA/1111 -> null)",
    f"Created {len(indicator_cols_created)} missingness indicator features",
    f"Log-transformed {len(LOG_TRANSFORM_COLS)} skewed numeric features",
    f"Clipped outliers (CLTV <=200, interest_rate <=30)",
    f"Engineered: loan_to_income_ratio, is_joint_application, high_cost_flag",
    f"Encoded {len(CAT_ONEHOT)} OneHot + {len(CAT_INDEX_ONLY)} Index-only categoricals",
    f"Built {len(pipeline.getStages())}-stage PySpark ML Pipeline",
    f"Stratified 80/20 split: {train_count:,} train / {test_count:,} test",
    f"Feature vector: {train_dim} dimensions",
    f"Pipeline model saved for production use",
    "",
    "--- Key Design Decisions ---",
    "  Drop 12 leakage columns      | Empirically confirmed EDA Cell 21",
    "  Log-transform financial cols  | Skewness > 2 in EDA Cell 4",
    "  Median imputation (not mean)  | Heavy right skew in financial data",
    "  OneHot <=10, Index 11-50      | Balance model compat vs dimensions",
    "  StandardScaler withMean=False | Sparse + centering = dense = OOM",
    "  Pipeline (not manual)         | Prevents leakage, serializable",
    "  Stratified split (not random) | Preserve ~80/20 imbalance in split",
    "",
    "--- BUG FIXES (v2) ---",
    "  Column name normalization     | hyphen→underscore BEFORE drops",
    "  Hard assert on leakage drops  | Fail loudly, not silently",
    "  Post-drop leakage scan        | Belt-and-suspenders verification",
    "",
    "--- NEXT: Notebook 4 - Model Training & Evaluation ---",
    "  Load saved train/test Parquet",
    "  Train: Logistic Regression, Random Forest, GBT",
    "  Hyperparameter tuning: CrossValidator + ParamGridBuilder",
    "  Evaluate: ROC-AUC, PR-AUC, F1, confusion matrix",
    "  Fair lending audit: model bias by demographic group",
]
for line in lines:
    print(line)

# Don't stop Spark -- Notebook 4 will use it
# spark.stop()

NOTEBOOK 3 COMPLETE: Feature Engineering & Pipeline Construction

Started with 11,483,889 rows x 99 columns
Normalized column names (hyphens → underscores)
Filtered to 7,686,484 rows (action_taken in {1, 3})
Dropped 58 columns (leakage, IDs, NZV, redundant)
  ✓ Post-drop leakage assertion PASSED
Normalized HMDA missing values (Exempt/NA/1111 -> null)
Created 9 missingness indicator features
Log-transformed 4 skewed numeric features
Clipped outliers (CLTV <=200, interest_rate <=30)
Engineered: loan_to_income_ratio, is_joint_application, high_cost_flag
Encoded 22 OneHot + 3 Index-only categoricals
Built 29-stage PySpark ML Pipeline
Stratified 80/20 split: 6,148,713 train / 1,533,840 test
Feature vector: 145 dimensions
Pipeline model saved for production use

--- Key Design Decisions ---
  Drop 12 leakage columns      | Empirically confirmed EDA Cell 21
  Log-transform financial cols  | Skewness > 2 in EDA Cell 4
  Median imputation (not mean)  | Heavy right skew in financial data
  OneHo

## Design Decision Summary

| Decision | Why | Alternative |
|:---|:---|:---|
| **Normalize column names first** | Parquet/Spark may sanitize hyphens → underscores; drop logic requires exact match | Regex-based drops (fragile) |
| Drop 12 leakage columns | Fill-rate differs by class (EDA Cell 21) | Keep for post-hoc analysis only |
| **Assert leakage drops succeed** | Silent failures caused 4 denial_reason columns to leak into the original model | Trust "not found = OK" (WRONG) |
| **Post-drop leakage scan** | Defense-in-depth: pattern-match remaining columns against known leakage names | Single-point verification only |
| Log-transform loan/income/property | Skewness >> 1 in EDA Cell 4 | Box-Cox (more complex, marginal gain) |
| Median imputation | Robust to heavy right skew | KNN imputation (too slow for 10M rows) |
| OneHot ≤10, Index 11-50 | Balances model compatibility with dimensionality | Target encoding (leakage risk without Pipeline) |
| loan_to_income_ratio | Core underwriting metric: affordability | Could use DTI ratio (but it's a binned string) |
| Missingness indicators | EDA Cell 20: missingness predicts denial | Ignore missing patterns (loses free signal) |
| StandardScaler withMean=False | OneHotEncoder → sparse; centering → dense → OOM | Skip scaling (only OK for tree models) |
| Pipeline (not manual) | Prevents leakage; serializable for production | Manual = simpler but dangerous at scale |
| Stratified split | Preserve ~80/20 class imbalance | Random split (acceptable for balanced data only) |

---

**NEXT → Notebook 4: Model Training & Evaluation**